# Notebook 4: The Bias-Variance Tradeoff

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours  
**Week 5 — ML Fundamentals & Linear Regression**

---

## What You Will Learn

| Concept | One-line Summary |
|---|---|
| Bias | How systematically wrong your model is on average |
| Variance | How much your model's predictions change with different training data |
| Bias-Variance Decomposition | Total Error = Bias² + Variance + Irreducible Noise |
| Underfitting | High bias — too simple to capture the pattern |
| Overfitting | High variance — too complex, memorises noise |
| The Sweet Spot | Optimal complexity that minimises total error |

## 1. Why This Matters

The bias-variance tradeoff is the **central tension in all of machine learning**.  
Every modelling decision you make — choosing model complexity, the amount of regularisation, the number of training examples — affects where you sit on this tradeoff.

Understanding it formally lets you:
- Diagnose *why* a model is failing (is it too simple? too complex?)
- Choose the right fix (add data? simplify the model? regularise?)
- Interpret learning curves and validation curves correctly

Without understanding bias and variance, you are flying blind.

## 2. Real-World Analogy — The Archery Range

Imagine two archery students practising at a target:

| | **Low Variance** | **High Variance** |
|---|---|---|
| **Low Bias** | Tight cluster at the centre ✓ | Scattered arrows, centred on bullseye |
| **High Bias** | Tight cluster, but always to the left ✗ | Scattered arrows, offset from bullseye |

**For a model:**
- **Bias** = how far the *average* prediction is from the true answer, across many training sets
- **Variance** = how much the prediction *changes* when you train on different datasets

**The goal:** small bias (aim is correct) AND small variance (shots are consistent).  
The problem: making a model more flexible usually reduces bias but increases variance — and vice versa.

> In house price prediction: bias means you systematically underestimate or overestimate prices. Variance means your model gives wildly different estimates depending on which 30 houses you happened to train on.

## 3. The Formal Decomposition

For a given test point x, the **Expected Mean Squared Error** of a model can be decomposed as:

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\left(\mathbb{E}[\hat{f}(x)] - f(x)\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[\left(\hat{f}(x) - \mathbb{E}[\hat{f}(x)]\right)^2\right]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Irreducible Noise}}$$

**Plain English:**

| Term | What it measures | Can we reduce it? |
|---|---|---|
| **Bias²** | How far the average prediction is from truth | Yes — use a more flexible model |
| **Variance** | How much predictions vary across training sets | Yes — use simpler model, more data, regularise |
| **Irreducible Noise σ²** | The inherent randomness in y that no model can explain | No |

This decomposition holds exactly for squared error loss and approximately for other losses.

In [ ]:
# ── Cell 1: Imports & Setup ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Fix the random seed for reproducibility across all cells
np.random.seed(42)

# Consistent figure style
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
print("Ready.")

## 4. High Bias (Underfitting) — The Model Is Too Simple

**Underfitting** happens when the model's hypothesis space is too restricted to capture the true pattern.  
Think of trying to fit a straight line through data that follows a sine wave — the line cannot bend.

**Symptoms:**
- High training error (can't even fit the training set well)
- High validation error (obviously can't generalise)
- Training error ≈ validation error (the model is consistently wrong in the same way)

**Fixes:** more flexible model, add features, reduce regularisation

In [ ]:
# ── Cell 2: Generate True Function and Observe Underfitting ───────────────────
# True function: y = sin(2π x) — a clean wave with additive noise
# Analogy: seasonal price swings in a housing market (prices dip in winter, spike in summer)
n = 30
X_all = np.sort(np.random.uniform(0, 1, n))
noise_std = 0.25                                # Irreducible noise level
y_true    = np.sin(2 * np.pi * X_all)          # True (unknown) function
y_obs     = y_true + np.random.normal(0, noise_std, n)  # What we actually observe

# Fine grid for plotting smooth curves
x_plot = np.linspace(0, 1, 300)

# Degree-1: a straight line — clearly too simple for a sine wave
model_d1 = Pipeline([('poly', PolynomialFeatures(1)),
                     ('lr',   LinearRegression())])
model_d1.fit(X_all.reshape(-1, 1), y_obs)

# Degree-10: very flexible — can potentially fit the wave
model_d10 = Pipeline([('poly', PolynomialFeatures(10)),
                      ('lr',   LinearRegression())])
model_d10.fit(X_all.reshape(-1, 1), y_obs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, model, deg, col, label in [
    (axes[0], model_d1,  1,  '#E53935', 'Degree 1 — UNDERFIT (High Bias)'),
    (axes[1], model_d10, 10, '#F44336', 'Degree 10 — OVERFIT (High Variance)')
]:
    y_fit = model.predict(x_plot.reshape(-1, 1))
    mse_t = mean_squared_error(y_obs, model.predict(X_all.reshape(-1, 1)))

    ax.scatter(X_all, y_obs, color='gray', s=30, zorder=5, label='Training data')
    ax.plot(x_plot, np.sin(2 * np.pi * x_plot), 'k--', lw=1.5,
            alpha=0.6, label='True sin(2πx)')
    ax.plot(x_plot, y_fit, color=col, lw=2.5, label=f'Train MSE={mse_t:.3f}')
    ax.set(title=label, xlabel='x', ylabel='y', ylim=(-2, 2))
    ax.legend(fontsize=9)

plt.suptitle('Underfitting vs Overfitting on sin(2πx) Data', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Empirical Bootstrap Experiment — Seeing Bias and Variance Directly

The formal definitions involve an expectation over **many training datasets** — something we normally never have.  
We can simulate this with **bootstrapping**: draw many independent training sets from the same distribution, fit the model on each, and observe how predictions vary.

- **Bias** = (mean prediction across all bootstrap models) - (true value) — squared
- **Variance** = spread of predictions across all bootstrap models

Let us run this experiment for a degree-1 (high bias) and degree-10 (high variance) model.

In [ ]:
# ── Cell 3: Bootstrap Experiment Setup ────────────────────────────────────────
n_bootstrap = 200     # Number of independent training sets to simulate
n_train     = 30      # Size of each training set
noise_std   = 0.25    # Same noise level as before

# A fine grid of test points where we evaluate bias and variance
x_test  = np.linspace(0, 1, 100)
y_truth = np.sin(2 * np.pi * x_test)   # The true noiseless function

# Store all predictions: shape (n_bootstrap, n_test_points)
preds_d1  = np.zeros((n_bootstrap, len(x_test)))
preds_d10 = np.zeros((n_bootstrap, len(x_test)))

for b in range(n_bootstrap):
    # Draw a fresh training set each iteration (different random seed)
    X_b = np.sort(np.random.uniform(0, 1, n_train))
    y_b = np.sin(2 * np.pi * X_b) + np.random.normal(0, noise_std, n_train)

    # Fit degree-1 model on this bootstrap sample
    m1 = Pipeline([('poly', PolynomialFeatures(1)), ('lr', LinearRegression())])
    m1.fit(X_b.reshape(-1, 1), y_b)
    preds_d1[b] = m1.predict(x_test.reshape(-1, 1))

    # Fit degree-10 model on this bootstrap sample
    m10 = Pipeline([('poly', PolynomialFeatures(10)), ('lr', LinearRegression())])
    m10.fit(X_b.reshape(-1, 1), y_b)
    preds_d10[b] = m10.predict(x_test.reshape(-1, 1))

print(f"Bootstrap experiment complete: {n_bootstrap} training sets, {n_train} points each.")
print(f"Predictions matrix shape (degree-1):  {preds_d1.shape}")
print(f"Predictions matrix shape (degree-10): {preds_d10.shape}")

In [ ]:
# ── Cell 4: Plot ALL Bootstrap Fits — See Bias and Variance Visually ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, preds, deg, col, title_label in [
    (axes[0], preds_d1,  1,  '#2196F3',
     'Degree-1 (Linear) — HIGH BIAS, LOW VARIANCE\nAll lines are similar but all miss the wave'),
    (axes[1], preds_d10, 10, '#F44336',
     'Degree-10 — LOW BIAS, HIGH VARIANCE\nSome fits are great; many are wildly wrong')
]:
    # Draw first 80 individual bootstrap fits (semi-transparent)
    for b in range(80):
        ax.plot(x_test, preds[b], color=col, alpha=0.06, linewidth=0.8)

    # Mean prediction across all 200 bootstrap models
    mean_pred = preds.mean(axis=0)
    ax.plot(x_test, mean_pred, color=col, linewidth=2.5,
            label=f'Mean prediction (over {n_bootstrap} models)')

    # True noiseless function for reference
    ax.plot(x_test, y_truth, 'k--', linewidth=1.8, alpha=0.8, label='True sin(2πx)')

    ax.set(title=title_label, xlabel='x', ylabel='y', ylim=(-3.5, 3.5))
    ax.legend(fontsize=9)

plt.suptitle('Bootstrap Experiment: 200 Models Fitted on Different Training Sets',
             fontsize=12)
plt.tight_layout()
plt.show()

## 6. Computing Empirical Bias², Variance, and Total Error

Now we use the bootstrap predictions to compute these quantities numerically.

In [ ]:
# ── Cell 5: Numerical Bias-Variance Decomposition ─────────────────────────────
def bias_variance_decomposition(preds, truth, noise_var):
    """
    Compute empirical bias², variance, and total expected error.

    Parameters
    ----------
    preds     : array (n_bootstrap, n_test) — predictions across all bootstrap fits
    truth     : array (n_test,)             — true noiseless function values
    noise_var : float                       — known irreducible noise variance σ²

    Returns
    -------
    Dictionary with mean bias², mean variance, total error, and irreducible noise.
    """
    mean_pred = preds.mean(axis=0)              # Average prediction at each test point

    # Bias² at each test point: (average prediction - truth)²
    bias2 = (mean_pred - truth) ** 2

    # Variance at each test point: average squared deviation from the mean prediction
    variance = preds.var(axis=0)

    # Total expected error (should approximately equal bias² + variance + noise)
    total = bias2 + variance + noise_var

    return {
        'bias2':    bias2.mean(),
        'variance': variance.mean(),
        'noise':    noise_var,
        'total':    total.mean()
    }

noise_var = noise_std ** 2          # σ² = 0.25² = 0.0625

bv_d1  = bias_variance_decomposition(preds_d1,  y_truth, noise_var)
bv_d10 = bias_variance_decomposition(preds_d10, y_truth, noise_var)

# Print a clean comparison table
print("Bias-Variance Decomposition (averaged over all test points)")
print("-" * 60)
print(f"{'Component':<20} {'Degree-1':>12} {'Degree-10':>12}")
print("-" * 60)
for key in ['bias2', 'variance', 'noise', 'total']:
    label = {'bias2': 'Bias²', 'variance': 'Variance',
             'noise': 'Irreducible Noise', 'total': 'Total Error'}[key]
    print(f"{label:<20} {bv_d1[key]:>12.4f} {bv_d10[key]:>12.4f}")
print("-" * 60)
print(f"\nDegree-1:  HIGH bias, LOW  variance")
print(f"Degree-10: LOW  bias, HIGH variance")

In [ ]:
# ── Cell 6: Bias-Variance Decomposition Bar Chart ─────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

components  = ['Bias²', 'Variance', 'Irreducible Noise']
vals_d1     = [bv_d1['bias2'],  bv_d1['variance'],  bv_d1['noise']]
vals_d10    = [bv_d10['bias2'], bv_d10['variance'], bv_d10['noise']]

x_pos = np.arange(len(components))
width = 0.35
pal   = ['#2196F3', '#F44336']   # Blue = degree-1, red = degree-10

bars1 = ax.bar(x_pos - width/2, vals_d1,  width, label='Degree-1',  color=pal[0], alpha=0.85)
bars2 = ax.bar(x_pos + width/2, vals_d10, width, label='Degree-10', color=pal[1], alpha=0.85)

# Label each bar with its value for clarity
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.4f}',
                ha='center', va='bottom', fontsize=9)

ax.set(xticks=x_pos, xticklabels=components,
       ylabel='Error (MSE units)',
       title='Bias-Variance Decomposition\n'
             'Degree-1: dominated by bias  |  Degree-10: dominated by variance')
ax.legend()
plt.tight_layout()
plt.show()

## 7. The Classic U-Shaped Curve — Sweeping Model Complexity

The key insight is that as we increase model complexity (polynomial degree 1 → 15):
- **Bias²** monotonically decreases — the model can fit more complex patterns
- **Variance** monotonically increases — the model becomes more sensitive to which data it was trained on
- **Total Error** forms a U-shape — there is an optimal complexity in the middle

The **sweet spot** is at the bottom of the U — where bias and variance are balanced.

In [ ]:
# ── Cell 7: Sweep Polynomial Degree 1–15, Compute Bias² and Variance ──────────
degrees     = list(range(1, 16))  # Polynomial degrees to try
n_boot      = 200                 # Bootstrap samples per degree
n_tr        = 30                  # Training set size

bias2_list  = []
var_list    = []

for deg in degrees:
    preds_deg = np.zeros((n_boot, len(x_test)))

    for b in range(n_boot):
        X_b = np.sort(np.random.uniform(0, 1, n_tr))
        y_b = np.sin(2 * np.pi * X_b) + np.random.normal(0, noise_std, n_tr)

        # Clip degree to min(deg, n_tr-1) to avoid singular matrices
        safe_deg = min(deg, n_tr - 1)
        m = Pipeline([('poly', PolynomialFeatures(safe_deg)),
                      ('lr',   LinearRegression())])
        m.fit(X_b.reshape(-1, 1), y_b)
        preds_deg[b] = m.predict(x_test.reshape(-1, 1))

    # Compute bias² and variance averaged across test points
    mean_p = preds_deg.mean(axis=0)
    bias2_list.append(((mean_p - y_truth) ** 2).mean())
    var_list.append(preds_deg.var(axis=0).mean())

bias2_arr = np.array(bias2_list)
var_arr   = np.array(var_list)
total_arr = bias2_arr + var_arr + noise_var   # Total = B² + V + σ²

# Find the degree with minimum total error
best_deg = degrees[np.argmin(total_arr)]
print(f"Optimal polynomial degree: {best_deg}  (minimum total expected error)")

In [ ]:
# ── Cell 8: The Classic Bias-Variance U-Curve ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(degrees, bias2_arr, 'o-', color='#2196F3',  lw=2, ms=6,  label='Bias²')
ax.plot(degrees, var_arr,   's-', color='#F44336',  lw=2, ms=6,  label='Variance')
ax.plot(degrees, total_arr, 'D-', color='#212121',  lw=2.5, ms=7, label='Total Error (B²+V+σ²)')

# Horizontal dashed line for irreducible noise floor
ax.axhline(noise_var, color='gray', linestyle=':', lw=1.5,
           label=f'Irreducible Noise σ²={noise_var:.4f}')

# Vertical line marking the optimal complexity
ax.axvline(best_deg, color='green', linestyle='--', lw=1.8,
           label=f'Sweet Spot: Degree {best_deg}')

# Annotate key regions
ax.annotate('← HIGH BIAS\n(Underfitting)', xy=(2, total_arr[1]),
            xytext=(3, total_arr[1] + 0.08),
            fontsize=9, color='#2196F3',
            arrowprops=dict(arrowstyle='->', color='#2196F3'))
ax.annotate('HIGH VARIANCE →\n(Overfitting)', xy=(12, total_arr[11]),
            xytext=(9, total_arr[11] + 0.08),
            fontsize=9, color='#F44336',
            arrowprops=dict(arrowstyle='->', color='#F44336'))

ax.set(xlabel='Polynomial Degree (Model Complexity →)',
       ylabel='Expected Error (MSE)',
       title='The Bias-Variance Tradeoff: U-Shaped Total Error Curve\n'
             'Sweet spot balances bias and variance',
       xlim=(1, 15))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. High Bias: Symptoms and Diagnosis

**How to recognise underfitting (high bias) in practice:**

- Training error is HIGH (the model fails even on data it has seen)
- Validation error ≈ training error (both are large — the model is consistently wrong)
- Adding more training data barely helps

**Common causes in house price prediction:**
- Using only one feature (sq ft) when price also depends on location, age, bedrooms
- Forcing a linear model when the relationship is curved
- Over-regularising (very high λ in Ridge regression)

In [ ]:
# ── Cell 9: Underfitting — Learning Curves Show Both Errors Stay High ──────────
# We will train a degree-1 model with increasing amounts of data
# and track both training and validation error
np.random.seed(42)
N_total = 200

# True sinusoidal house price pattern
X_full_data = np.sort(np.random.uniform(0, 1, N_total))
y_full_data = np.sin(2 * np.pi * X_full_data) + np.random.normal(0, 0.25, N_total)

# Hold out 50 points as a fixed validation set
X_tr_pool, X_val_lc, y_tr_pool, y_val_lc = train_test_split(
    X_full_data, y_full_data, test_size=50, random_state=0)

train_sizes = range(5, len(X_tr_pool), 5)

def learning_curve(model_class, train_sizes, X_tr, y_tr, X_val, y_val):
    """Compute train and validation MSE for increasing training set sizes."""
    tr_errors, val_errors = [], []
    for n in train_sizes:
        m = model_class()           # Fresh model each time
        m.fit(X_tr[:n].reshape(-1, 1), y_tr[:n])
        tr_errors.append(mean_squared_error(y_tr[:n],
                         m.predict(X_tr[:n].reshape(-1, 1))))
        val_errors.append(mean_squared_error(y_val,
                          m.predict(X_val.reshape(-1, 1))))
    return np.array(tr_errors), np.array(val_errors)

# Degree-1: underfitting
class LinearDeg1:
    def __init__(self):
        self.model = Pipeline([('poly', PolynomialFeatures(1)),
                                ('lr', LinearRegression())])
    def fit(self, X, y): self.model.fit(X, y); return self
    def predict(self, X): return self.model.predict(X)

# Degree-5: reasonable fit for a sine wave
class LinearDeg5:
    def __init__(self):
        self.model = Pipeline([('poly', PolynomialFeatures(5)),
                                ('lr', LinearRegression())])
    def fit(self, X, y): self.model.fit(X, y); return self
    def predict(self, X): return self.model.predict(X)

tr_d1, val_d1 = learning_curve(LinearDeg1, train_sizes,
                                X_tr_pool, y_tr_pool, X_val_lc, y_val_lc)
tr_d5, val_d5 = learning_curve(LinearDeg5, train_sizes,
                                X_tr_pool, y_tr_pool, X_val_lc, y_val_lc)

sizes = list(train_sizes)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, tr_err, val_err, title in [
    (axes[0], tr_d1, val_d1, 'Degree-1 (High Bias)\nBoth errors stay high — adding data barely helps'),
    (axes[1], tr_d5, val_d5, 'Degree-5 (Good Fit)\nValidation error converges toward training error')
]:
    ax.plot(sizes, tr_err,  'o-', color='#2196F3', lw=2, ms=4, label='Train Error')
    ax.plot(sizes, val_err, 's-', color='#F44336', lw=2, ms=4, label='Validation Error')
    ax.set(title=title, xlabel='Training Set Size', ylabel='MSE')
    ax.legend(fontsize=9)

plt.suptitle('Learning Curves: Underfitting vs Good Fit', fontsize=12)
plt.tight_layout()
plt.show()

## 9. High Variance: Symptoms and Diagnosis

**How to recognise overfitting (high variance) in practice:**

- Training error is VERY LOW (the model memorises the training set)
- Validation error is MUCH HIGHER than training error (fails to generalise)
- As training size increases, the gap narrows significantly

**Common causes in house price prediction:**
- Too many features relative to data (100 features, 50 houses)
- Decision tree with no depth limit (memorises every house individually)
- Very small k in KNN (k=1 always memorises training set perfectly)

In [ ]:
# ── Cell 10: House Price — Underfitting vs Overfitting with Real-ish Data ──────
np.random.seed(42)
N = 150

# Simulate house prices with multiple non-linear features
sqft     = np.random.uniform(600, 3500, N)
age      = np.random.uniform(0, 60, N)     # House age in years

# True rule: price grows with size but decreases with age (non-linearly)
price_hp = (50 + 0.12 * sqft - 0.5 * age +
            0.0002 * sqft * (60 - age) +   # Interaction: new large houses = premium
            np.random.normal(0, 15, N))

X_hp = np.column_stack([sqft, age])
X_tr_hp, X_val_hp, y_tr_hp, y_val_hp = train_test_split(
    X_hp, price_hp, test_size=0.3, random_state=42)

# Model 1: linear (likely underfit — misses the interaction term)
lr_house = LinearRegression().fit(X_tr_hp, y_tr_hp)

# Model 2: decision tree, no depth limit (likely overfit on 105 training points)
dt_deep = DecisionTreeRegressor(max_depth=None, min_samples_leaf=1)
dt_deep.fit(X_tr_hp, y_tr_hp)

# Model 3: decision tree, limited depth (the Goldilocks model)
dt_good = DecisionTreeRegressor(max_depth=5, min_samples_leaf=5)
dt_good.fit(X_tr_hp, y_tr_hp)

models_hp = [
    ('Linear Regression\n(Underfit — High Bias)', lr_house),
    ('Decision Tree (Unlimited)\n(Overfit — High Variance)', dt_deep),
    ('Decision Tree (Depth=5)\n(Good Balance)', dt_good)
]

print(f"{'Model':<35} {'Train MSE':>12} {'Val MSE':>12}")
print("-" * 62)
for name, model in models_hp:
    tr_mse  = mean_squared_error(y_tr_hp,  model.predict(X_tr_hp))
    val_mse = mean_squared_error(y_val_hp, model.predict(X_val_hp))
    print(f"{name.split(chr(10))[0]:<35} {tr_mse:>12.1f} {val_mse:>12.1f}")

In [ ]:
# ── Cell 11: Validation Curves for Decision Tree Depth ────────────────────────
# Sweep max_depth from 1 to 20, track train and validation MSE
depths     = list(range(1, 21))
tr_mses_dt  = []
val_mses_dt = []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, min_samples_leaf=2, random_state=0)
    dt.fit(X_tr_hp, y_tr_hp)
    tr_mses_dt.append(mean_squared_error(y_tr_hp,  dt.predict(X_tr_hp)))
    val_mses_dt.append(mean_squared_error(y_val_hp, dt.predict(X_val_hp)))

best_depth = depths[np.argmin(val_mses_dt)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, tr_mses_dt,  'o-', color='#2196F3', lw=2, ms=5, label='Train MSE')
ax.plot(depths, val_mses_dt, 's-', color='#F44336', lw=2, ms=5, label='Validation MSE')
ax.axvline(best_depth, color='green', linestyle='--', lw=1.8,
           label=f'Optimal depth = {best_depth}')

# Shade the two failure regions
ax.axvspan(1, 3, alpha=0.08, color='blue', label='High Bias Zone')
ax.axvspan(12, 20, alpha=0.08, color='red', label='High Variance Zone')

ax.set(xlabel='Decision Tree Max Depth (Complexity →)',
       ylabel='MSE',
       title='House Price Prediction — Validation Curve\n'
             'Training error always drops; validation error makes a U-shape')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Best depth: {best_depth} | Val MSE at best depth: {min(val_mses_dt):.1f}")

## 10. The Archery Visualisation — Bias and Variance as Scatter Plots

Let us return to the archery analogy and make it concrete with a plot showing all four combinations.

In [ ]:
# ── Cell 12: Archery Target — Four Bias-Variance Scenarios ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

scenarios = [
    # (title,        center_offset, spread, color)
    ('Low Bias, Low Variance\n(The goal)',       (0.0,  0.0),  0.08, '#4CAF50'),
    ('Low Bias, High Variance\n(Random)',         (0.0,  0.0),  0.35, '#FF9800'),
    ('High Bias, Low Variance\n(Consistently wrong)', (-0.4, -0.3), 0.08, '#2196F3'),
    ('High Bias, High Variance\n(Worst case)',   (-0.35,-0.25), 0.35, '#F44336'),
]

for ax, (title, (cx, cy), spread, col) in zip(axes.flatten(), scenarios):
    # Draw concentric target rings
    for r, alpha in [(0.5, 0.08), (0.35, 0.12), (0.2, 0.18), (0.08, 0.3)]:
        circle = plt.Circle((0, 0), r, color='gray', fill=True, alpha=alpha)
        ax.add_patch(circle)

    # Bullseye
    ax.add_patch(plt.Circle((0, 0), 0.04, color='red', fill=True))

    # Simulate 30 shots: arrows land near (cx, cy) with given spread
    xs = np.random.normal(cx, spread, 30)
    ys = np.random.normal(cy, spread, 30)

    ax.scatter(xs, ys, color=col, s=40, zorder=5, alpha=0.8, edgecolors='white', lw=0.5)

    # Mark the mean shot position
    ax.scatter([xs.mean()], [ys.mean()], color='black', marker='X', s=120,
               zorder=6, label='Mean shot')

    ax.set(xlim=(-0.8, 0.8), ylim=(-0.8, 0.8),
           title=title, aspect='equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Archery Analogy: Four Bias-Variance Scenarios\nBlack X = mean prediction | Red dot = bullseye (truth)',
             fontsize=12)
plt.tight_layout()
plt.show()

## 11. Irreducible Noise — What No Model Can Fix

The σ² term in the decomposition is **irreducible** — it represents the inherent randomness in the outcome variable that no model, however perfect, can predict.

In house price prediction, irreducible noise comes from:
- Negotiation outcomes (two identical houses, different buyers, different final prices)
- Unmeasured factors (seller urgency, buyer emotion, inspection surprises)
- Micro-location effects (one house is next to a noisy HVAC unit, identical spec otherwise)
- Measurement error in recorded prices

**The practical implication:** there is a minimum achievable MSE on any real-world problem. Chasing a model that achieves zero training error is chasing a mirage — you are fitting the noise.

In [ ]:
# ── Cell 13: Visualising the Noise Floor ──────────────────────────────────────
np.random.seed(42)
x_demo = np.linspace(0, 1, 200)

# True function — what we are trying to learn
f_true = np.sin(2 * np.pi * x_demo)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
noise_levels = [0.05, 0.25, 0.6]

for ax, sigma in zip(axes, noise_levels):
    y_noisy = f_true + np.random.normal(0, sigma, len(x_demo))
    ax.plot(x_demo, f_true,  'k-',  lw=2, label='True f(x)')
    ax.scatter(x_demo, y_noisy, color='steelblue', s=8, alpha=0.5, label='Observed y')
    ax.fill_between(x_demo, f_true - 2*sigma, f_true + 2*sigma,
                    alpha=0.15, color='gray', label=f'±2σ band')
    ax.set(title=f'σ = {sigma}  |  σ² = {sigma**2:.4f}\n'
                 f'(Irreducible noise floor: {sigma**2:.4f})',
           xlabel='x')
    ax.legend(fontsize=8)

axes[0].set_ylabel('y')
plt.suptitle('Irreducible Noise (σ²): The Minimum Achievable Error\n'
             'No model can predict the noise — only σ² matters here', fontsize=11)
plt.tight_layout()
plt.show()

## 12. Summary — The Complete Picture

```
Total Expected Error = Bias² + Variance + Irreducible Noise
```

| Scenario | Bias² | Variance | Fix |
|---|---|---|---|
| Underfitting | High | Low | More complex model, more features, less regularisation |
| Overfitting | Low | High | More data, regularisation, simpler model, dropout |
| Just right | Low | Low | You are done — focus on irreducible noise |

### Fixes Cheatsheet

| Problem | Lever | Effect |
|---|---|---|
| High Bias | Increase model complexity | ↓ Bias, ↑ Variance |
| High Bias | Add more features | ↓ Bias |
| High Variance | Collect more training data | ↓ Variance |
| High Variance | Regularisation (Ridge/Lasso) | ↓ Variance, slight ↑ Bias |
| High Variance | Reduce model complexity | ↓ Variance, slight ↑ Bias |
| High Variance | Ensemble methods (Random Forest) | ↓ Variance, maintain Bias |

### The Diagnostic Protocol

```
1. Compute training error and validation error
2. If training error is high → High Bias → increase complexity
3. If training error is low but val error is high → High Variance → regularise / get more data
4. If both are low → celebrate (but check on test set!)
```

In [ ]:
# ── Cell 14: Final Summary Plot — Everything on One Figure ────────────────────
fig = plt.figure(figsize=(14, 9))
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# ── Top row: the classic decomposition curves ──────────────────────────────────
ax_curve = fig.add_subplot(gs[0, :])
ax_curve.plot(degrees, bias2_arr,  'o-', color='#2196F3', lw=2, ms=5, label='Bias²')
ax_curve.plot(degrees, var_arr,    's-', color='#F44336', lw=2, ms=5, label='Variance')
ax_curve.plot(degrees, total_arr,  'D-', color='#212121', lw=2.5, ms=6,
              label='Total Error')
ax_curve.axhline(noise_var, color='gray', linestyle=':', lw=1.5, label='Noise floor σ²')
ax_curve.axvline(best_deg, color='green', linestyle='--', lw=1.8,
                 label=f'Sweet Spot (degree {best_deg})')
ax_curve.set(xlabel='Polynomial Degree', ylabel='Expected MSE',
             title='The Bias-Variance Tradeoff — Comprehensive View')
ax_curve.legend(fontsize=9, loc='upper right')

# ── Bottom row: three representative fits ─────────────────────────────────────
example_degrees = [1, best_deg, 12]
ex_titles = [
    f'Degree 1\n(HIGH BIAS — Underfit)',
    f'Degree {best_deg}\n(SWEET SPOT)',
    'Degree 12\n(HIGH VARIANCE — Overfit)'
]
ex_colors = ['#2196F3', '#4CAF50', '#F44336']

# Use a single training sample to show a concrete fit
np.random.seed(10)
X_ex = np.sort(np.random.uniform(0, 1, 30))
y_ex = np.sin(2 * np.pi * X_ex) + np.random.normal(0, 0.25, 30)
x_plot_ex = np.linspace(0, 1, 300)

for col_idx, (deg, title, col) in enumerate(zip(example_degrees, ex_titles, ex_colors)):
    ax = fig.add_subplot(gs[1, col_idx])
    m  = Pipeline([('poly', PolynomialFeatures(deg)),
                   ('lr',   LinearRegression())])
    m.fit(X_ex.reshape(-1, 1), y_ex)
    y_fit   = m.predict(x_plot_ex.reshape(-1, 1))
    mse_val = mean_squared_error(y_ex, m.predict(X_ex.reshape(-1, 1)))

    ax.scatter(X_ex, y_ex, color='gray', s=20, zorder=5)
    ax.plot(x_plot_ex, np.sin(2 * np.pi * x_plot_ex),
            'k--', lw=1.5, alpha=0.6, label='Truth')
    ax.plot(x_plot_ex, y_fit, color=col, lw=2.5, label=f'MSE={mse_val:.3f}')
    ax.set(title=title, xlabel='x', ylim=(-2.5, 2.5))
    if col_idx == 0: ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.savefig('bias_variance_summary.png', bbox_inches='tight', dpi=100)
plt.show()
print("Summary figure saved.")

## 13. Self-Check Questions

Test your understanding. Try to answer each question before expanding.

---

**Question 1:** A KNN model with k=1 achieves 0% training error. What does this tell you about its bias and variance?

<details>
<summary>Click to reveal answer</summary>

With k=1, the model assigns the prediction for any training point to be exactly that point's own label — so training error is always zero by definition. This tells us:

- **Bias ≈ 0 on training data** — the model makes no systematic errors on the data it has seen.
- **Very high variance** — since the prediction for each training point is just that one point's value, the model is extremely sensitive to the specific training set. Change one training point and the prediction map changes dramatically in that region.

On new data (validation set), k=1 will have **high variance** — it memorises noise rather than the underlying pattern. This is a textbook case of overfitting. Increasing k smooths the predictions (more neighbours → less sensitivity to individual points → lower variance, but higher bias).

</details>

---

**Question 2:** You add 10x more training data. Which decreases more: bias or variance? Why?

<details>
<summary>Click to reveal answer</summary>

**Variance decreases more.** Here is the reasoning:

- **Bias** is determined by the model's hypothesis space and inductive bias — these are independent of how much data you have. A linear model on a sinusoidal dataset will still have high bias with 10,000 training points, because it simply cannot express a curved function.
- **Variance** decreases as you add more data because the model has more information to work with, so it becomes less sensitive to any particular training set. With more data, the "wiggliness" of a complex model gets constrained — there are fewer ways to overfit 10,000 points than 1,000 points.

**Rule of thumb:** More data is the best cure for high variance. It does almost nothing to cure high bias.

</details>

---

**Question 3:** Irreducible noise cannot be reduced by any model. Where does this noise come from in the house price prediction context?

<details>
<summary>Click to reveal answer</summary>

In house price prediction, irreducible noise σ² arises from factors that are either unobservable or inherently random:

1. **Negotiation dynamics:** Two identical houses sell for different prices because one seller was in a hurry (divorce, job relocation) and accepted a lower offer.
2. **Buyer subjectivity:** One buyer loved the seller's renovated kitchen and paid a premium; another buyer planned to renovate anyway and did not.
3. **Unmeasured micro-features:** A house is next to a noisy HVAC unit on the adjacent commercial building — this is not in any dataset.
4. **Inspection surprises:** A hidden plumbing problem discovered during inspection knocked $15,000 off a deal.
5. **Timing:** Interest rates moved 0.5% between two otherwise identical transactions in the same month.
6. **Data recording errors:** The recorded sale price sometimes excludes personal property deals (furniture, appliances) that were bundled in.

Even a perfect model — one that knew the true function f(sqft, location, age, bedrooms, ...) exactly — would still make errors equal to σ² because these factors are inherently unknowable from available features.

</details>

---

**Question 4:** You plot training error and validation error vs model complexity. Training error always decreases. Validation error makes a U-shape. Where is the optimal model complexity?

<details>
<summary>Click to reveal answer</summary>

The optimal model complexity is at the **minimum of the validation error curve** — the bottom of the U-shape.

Here is what the curve tells you:

- **Left side (low complexity):** Both training and validation error are high — this is the high-bias (underfitting) region. The model is too simple.
- **Bottom of the U:** Validation error is minimised — this is the sweet spot where bias and variance are balanced. This is where you want to be.
- **Right side (high complexity):** Training error keeps decreasing but validation error rises — this is the high-variance (overfitting) region. The model is memorising training noise.

**Practical note:** In practice, you cannot compute the theoretical bias-variance decomposition, but you can always plot training vs validation error against model complexity — this gives you the same U-shaped guide to the optimal complexity. Use cross-validation to get a stable estimate of validation error.

</details>